# 🎬 Video Upscaler Pro no Google Colab (Tesla T4)
### Super-Resolução Neural (1080p / 4K) com Áudio Sincronizado para Clipes de 10s a 30s+

Este notebook é uma ferramenta autônoma dedicada ao **upscaling de alta fidelidade de vídeos**:
* 🚀 **Projetado para clipes de 10s a 30s+:** Processamento ultrarrápido na GPU Tesla T4 (~1 a 3 minutos por vídeo).
* 📁 **Modo Arquivo Único ou Modo Pasta (Lote):** Processe um único clipe ou aponte para uma pasta inteira no Google Drive e deixe ele processar toda a sua fila automaticamente.
* 🧠 **Modelos Top de Linha:** `4x-UltraSharp` (máxima nitidez fotorrealista) e `RealESRGAN_x4plus`.
* 🎵 **Áudio Estéreo 100% Sincronizado:** Isola e remonta as trilhas de som sem perda de sincronia.
* 💾 **Integração Total com o Google Drive:** Os modelos ficam salvos permanentemente e os vídeos finais são gravados direto no seu Drive.


In [ ]:
# =============================================================================
# 1. Verificação de GPU e Montagem do Google Drive
# =============================================================================
import os
import sys
import subprocess
import shutil
from pathlib import Path
import torch

print("🔍 1. Verificando Hardware...")
if not torch.cuda.is_available():
    raise RuntimeError("❌ Nenhuma GPU detectada! Vá em Ambiente de execução -> Alterar tipo de ambiente de execução -> GPU T4.")
gpu_name = torch.cuda.get_device_name(0)
print(f"   GPU Ativa: {gpu_name} (VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB)")

print("\n📁 2. Conectando ao Google Drive...")
DRIVE_MOUNT = Path("/content/drive")
if not (DRIVE_MOUNT / "MyDrive").is_dir():
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))

MY_DRIVE = DRIVE_MOUNT / "MyDrive"
print("   Google Drive montado com sucesso!")

print("\n📦 3. Instalando dependências necessárias (spandrel, opencv, tqdm)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spandrel", "opencv-python", "tqdm", "huggingface_hub"], check=True)
print("✅ Ambiente pronto para Upscaling!")


In [ ]:
# =============================================================================
# 2. Gerenciador de Modelos de Super-Resolução no Google Drive
# =============================================================================
# @title 📦 Baixar e Verificar Todos os Modelos no Google Drive { run: "auto" }
BAIXAR_TODOS_MODELOS = True  # @param {type:"boolean"}

drive_candidates = [
    MY_DRIVE / "ComfyUI" / "models" / "upscale_models",
    MY_DRIVE / "ComfyUi" / "models" / "upscale_models",
    MY_DRIVE / "comfyui" / "models" / "upscale_models",
    MY_DRIVE / "models" / "upscale_models",
]
UPSCALE_DIR = next((p for p in drive_candidates if p.is_dir()), MY_DRIVE / "ComfyUI" / "models" / "upscale_models")
UPSCALE_DIR.mkdir(parents=True, exist_ok=True)

# Catálogo completo dos 4 melhores modelos de super-resolução
UPSCALE_CATALOG = {
    "4xNomos8kDAT": {
        "file": "4xNomos8kDAT.pth",
        "repo": "Maxivi/SDXLModels",
        "bytes": 309328227,
        "desc": "🌟 4xNomos8kDAT (Vision Transformer DAT - O Campeão de Fotorrealismo e Texturas)"
    },
    "4x_NMKD-Superscale": {
        "file": "4x_NMKD-Superscale-SP_178000_G.pth",
        "repo": "gemasai/4x_NMKD-Superscale-SP_178000_G",
        "bytes": 67021162,
        "desc": "🎬 4x_NMKD-Superscale (ESRGAN Moderno - Peles Suaves e Equilíbrio Sem Forçar Bordas)"
    },
    "4x-UltraSharp": {
        "file": "4x-UltraSharp.pth",
        "repo": "lokcx/4x-Ultrasharp",
        "bytes": 66986421,
        "desc": "⚡ 4x-UltraSharp (ESRGAN Clássico - Máxima Nitidez e Contraste Acentuado)"
    },
    "RealESRGAN_x4plus": {
        "file": "RealESRGAN_x4plus.pth",
        "repo": "lllyasviel/Annotators",
        "bytes": 67021162,
        "desc": "🛡️ RealESRGAN_x4plus (ESRGAN Oficial - Limpeza de Artefatos e Ruídos)"
    }
}


def find_upscale_model_on_drive(filename, min_bytes=0):
    """Varre todas as pastas possíveis no Google Drive antes de qualquer download."""
    direct_candidates = [
        UPSCALE_DIR / filename,
        MY_DRIVE / "ComfyUI" / "models" / "upscale_models" / filename,
        MY_DRIVE / "ComfyUi" / "models" / "upscale_models" / filename,
        MY_DRIVE / "comfyui" / "models" / "upscale_models" / filename,
        MY_DRIVE / "models" / "upscale_models" / filename,
        MY_DRIVE / "upscale_models" / filename,
        Path("/content/ComfyUI/models/upscale_models") / filename,
        Path("/content/models/upscale_models") / filename,
    ]
    for candidate in direct_candidates:
        try:
            if candidate.is_file() and (min_bytes == 0 or candidate.stat().st_size >= min_bytes * 0.95):
                return candidate
        except Exception:
            pass

    for search_root in [MY_DRIVE / "ComfyUI", MY_DRIVE / "ComfyUi", MY_DRIVE / "H3", MY_DRIVE / "models"]:
        try:
            if search_root.is_dir():
                for match in search_root.glob(f"**/{filename}"):
                    if match.is_file() and (min_bytes == 0 or match.stat().st_size >= min_bytes * 0.95):
                        return match
        except Exception:
            pass
    return None

print(f"📂 Diretório Principal de Modelos no Google Drive: {UPSCALE_DIR}")
print("🔍 Verificando status dos modelos...")

from huggingface_hub import hf_hub_download

RESOLVED_MODEL_PATHS = {}

for key, info in UPSCALE_CATALOG.items():
    existing = find_upscale_model_on_drive(info["file"], info["bytes"])
    if existing:
        print(f"  [✅ JÁ EXISTE NO DRIVE] {info['desc']}")
        print(f"      -> Localizado em: {existing} ({existing.stat().st_size / (1024**2):.1f} MB)")
        RESOLVED_MODEL_PATHS[key] = existing
    elif BAIXAR_TODOS_MODELOS:
        print(f"  [⬇️ BAIXANDO PARA O DRIVE] {info['desc']}...")
        dest = UPSCALE_DIR / info["file"]
        try:
            downloaded = hf_hub_download(repo_id=info["repo"], filename=info["file"], local_dir=str(UPSCALE_DIR))
            p = Path(downloaded)
            if p != dest and p.is_file():
                shutil.move(str(p), str(dest))
            RESOLVED_MODEL_PATHS[key] = dest
            print(f"  [✅ CONCLUÍDO] {info['file']} salvo permanentemente no Drive!")
        except Exception as e:
            print(f"  [⚠️ Erro ao baixar {info['file']}]: {e}")
    else:
        print(f"  [⚪ DISPONÍVEL PARA BAIXAR SOB DEMANDA] {info['desc']}")

print("\n🎉 Modelos verificados e prontos para uso!")


## 🚀 Executar Upscaling de Vídeos (Individual ou Fila em Lote)

Configure os parâmetros abaixo no formulário:
* **`MODO_EXECUCAO`:**
  * `Arquivo_Unico`: Processa o vídeo indicado em `CAMINHO_VIDEO`.
  * `Fila_Em_Lote_Pasta`: Processa todos os arquivos `.mp4` dentro de `PASTA_DE_ENTRADA`.
* **`MODELO`:** Selecione `4x-UltraSharp` para máximo realismo.
* **`RESOLUCAO_FINAL`:** Escolha `1080p (Full HD)` (ideal para internet/redes), `2x` ou `4x`.
* **`PRESERVAR_AUDIO`:** Mantém a trilha de áudio original perfeitamente sincronizada.


In [ ]:
# =============================================================================
# 3. Processador de Vídeo com Super-Resolução Neural e Áudio
# =============================================================================
# @title 🎬 Configuração e Execução do Upscale { run: "auto" }
import os
import sys
import time
import shutil
import subprocess
from pathlib import Path
import cv2
import torch
from tqdm.auto import tqdm
from spandrel import ModelLoader
from base64 import b64encode
from IPython.display import HTML, display

# --- Menu Interativo de Seleção ---
MODO_EXECUCAO = "Arquivo_Unico"  # @param ["Arquivo_Unico", "Fila_Em_Lote_Pasta"]
CAMINHO_VIDEO = "/content/drive/MyDrive/H3/MiniMax-H3-Native-Colab/video"  # @param {type:"string"}
PASTA_DE_ENTRADA = "/content/drive/MyDrive/Videos_Para_Upscale"  # @param {type:"string"}
PASTA_DE_SAIDA = "/content/drive/MyDrive/Videos_Upscaled"  # @param {type:"string"}

# @markdown ### 🧠 Escolha o Modelo de Super-Resolução:
MODELO_ESCOLHIDO = "4x_NMKD-Superscale (Recomendado - Peles, Cinema e Alta Velocidade)"  # @param ["4x_NMKD-Superscale (Recomendado - Peles, Cinema e Alta Velocidade)", "4x-UltraSharp (Nitidez e Contraste Alto)", "RealESRGAN_x4plus (Reducao de Ruido)", "4xNomos8kDAT (Ultra-Lento - Recomendado para Fotos)"]
RESOLUCAO_FINAL = "1080p (Full HD)"  # @param ["1080p (Full HD)", "2x", "4x"]
PRESERVAR_AUDIO = True  # @param {type:"boolean"}
QUALIDADE_CRF = 17  # @param {type:"slider", min:14, max:24, step:1}

# Mapeamento do nome amigável para chave do catálogo
model_key_map = {
    "4x_NMKD-Superscale (Recomendado - Peles, Cinema e Alta Velocidade)": "4x_NMKD-Superscale",
    "4xNomos8kDAT (Ultra-Lento - Recomendado para Fotos)": "4xNomos8kDAT",
    "4x-UltraSharp (Nitidez e Contraste Alto)": "4x-UltraSharp",
    "RealESRGAN_x4plus (Reducao de Ruido e Compressao)": "RealESRGAN_x4plus",
}
active_model_key = model_key_map.get(MODELO_ESCOLHIDO, "4xNomos8kDAT")

# 1. Configurar pastas de saída
OUTPUT_DIR = Path(PASTA_DE_SAIDA)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Coletar lista de vídeos para processar
videos_to_process = []
if MODO_EXECUCAO == "Arquivo_Unico":
    target = Path(CAMINHO_VIDEO)
    if target.is_dir():
        mp4s = list(target.glob("*.mp4"))
        if mp4s:
            latest = max(mp4s, key=lambda p: p.stat().st_mtime)
            videos_to_process.append(latest)
            print(f"ℹ️ Caminho aponta para diretório. Selecionado o vídeo mais recente: {latest.name}")
        else:
            raise RuntimeError(f"Nenhum arquivo .mp4 encontrado em {target}")
    elif target.is_file():
        videos_to_process.append(target)
    else:
        raise RuntimeError(f"Arquivo ou pasta não encontrada: {target}")
else:
    in_dir = Path(PASTA_DE_ENTRADA)
    if not in_dir.is_dir():
        in_dir.mkdir(parents=True, exist_ok=True)
        print(f"📁 Pasta de entrada criada em: {in_dir}")
        print("   Coloque seus vídeos de 10s a 30s dentro dessa pasta no seu Drive e execute a célula novamente.")
    else:
        mp4s = sorted(list(in_dir.glob("*.mp4")))
        if not mp4s:
            print(f"⚠️ Nenhum vídeo .mp4 encontrado em {in_dir}")
        videos_to_process = mp4s

if not videos_to_process:
    print("Nenhum vídeo para processar no momento.")
else:
    print(f"🎯 Total de vídeos na fila: {len(videos_to_process)}")
    for i, v in enumerate(videos_to_process, 1):
        print(f"   [{i}] {v.name} ({v.stat().st_size / (1024**2):.1f} MiB)")

    # 3. VERIFICAÇÃO E CARREGAMENTO DO MODELO (Drive + Cache em VRAM)
    model_spec = UPSCALE_CATALOG[active_model_key]
    model_file = None
    if "RESOLVED_MODEL_PATHS" in globals() and active_model_key in globals()["RESOLVED_MODEL_PATHS"]:
        model_file = globals()["RESOLVED_MODEL_PATHS"][active_model_key]
    elif "find_upscale_model_on_drive" in globals():
        model_file = find_upscale_model_on_drive(model_spec["file"], model_spec["bytes"])
    
    if not model_file or not model_file.is_file():
        # Se ainda não foi baixado, baixa sob demanda para o Drive
        print(f"⬇️ Baixando {model_spec['file']} para o seu Google Drive...")
        from huggingface_hub import hf_hub_download
        downloaded = hf_hub_download(repo_id=model_spec["repo"], filename=model_spec["file"], local_dir=str(UPSCALE_DIR))
        model_file = Path(downloaded)
        dest = UPSCALE_DIR / model_spec["file"]
        if model_file != dest and model_file.is_file():
            shutil.move(str(model_file), str(dest))
            model_file = dest

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if "LOADED_UPSCALE_MODEL" in globals() and globals().get("CURRENT_MODEL_KEY") == active_model_key:
        print(f"\n⚡ [CACHE VRAM ATIVO] Modelo {active_model_key} já está carregado na GPU! Reutilizando instantaneamente...")
        upscale_model = globals()["LOADED_UPSCALE_MODEL"]
    else:
        print(f"\n🧠 Carregando {model_spec['desc']} na GPU Tesla T4...")
        print(f"   Arquivo: {model_file} ({model_file.stat().st_size / (1024**2):.1f} MB)")
        loader = ModelLoader()
        upscale_model = loader.load_from_file(str(model_file)).eval().to(device)
        model_dtype = torch.float32
        if hasattr(upscale_model, "supports_half") and upscale_model.supports_half:
            try:
                upscale_model = upscale_model.half()
                model_dtype = torch.float16
            except Exception:
                pass
        globals()["LOADED_UPSCALE_MODEL"] = upscale_model
        globals()["UPSCALE_MODEL_DTYPE"] = model_dtype
        globals()["CURRENT_MODEL_KEY"] = active_model_key
        print("   ✅ Modelo carregado com sucesso na VRAM da GPU!")

    # 4. Processar cada vídeo da fila
    for v_idx, video_path in enumerate(videos_to_process, 1):
        print("\n" + "=" * 76)
        print(f"🎬 Processando [{v_idx}/{len(videos_to_process)}]: {video_path.name}")
        print("=" * 76)
        
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            print(f"❌ Erro ao abrir {video_path.name}, pulando...")
            continue
            
        in_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        in_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration_s = total_frames / fps
        
        # Calcular dimensões de saída
        if RESOLUCAO_FINAL == "2x":
            out_w, out_h = in_w * 2, in_h * 2
        elif RESOLUCAO_FINAL == "4x":
            out_w, out_h = in_w * 4, in_h * 4
        else:  # 1080p
            out_h = 1080
            out_w = int(round((in_w * (1080 / in_h)) / 2) * 2)
            
        print(f"📊 Original: {in_w}x{in_h} @ {fps:.1f} fps ({duration_s:.1f}s | {total_frames} frames)")
        print(f"🚀 Saída   : {out_w}x{out_h} ({RESOLUCAO_FINAL}) | Modelo: {active_model_key}")
        
        temp_audio = Path(f"/content/temp_audio_{v_idx}.aac")
        has_audio = False
        if PRESERVAR_AUDIO:
            try:
                subprocess.run([
                    "ffmpeg", "-y", "-i", str(video_path), "-vn", "-c:a", "copy", str(temp_audio)
                ], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
                if temp_audio.is_file() and temp_audio.stat().st_size > 1000:
                    has_audio = True
                    print(f"🎵 Trilha de áudio estéreo preservada!")
            except Exception as e:
                print(f"⚠️ Aviso na extração de áudio: {e}")
                
        out_filename = f"{video_path.stem}_upscaled_{active_model_key}_{out_w}x{out_h}.mp4"
        out_path = OUTPUT_DIR / out_filename
        
        ffmpeg_cmd = [
            "ffmpeg", "-y",
            "-f", "rawvideo",
            "-vcodec", "rawvideo",
            "-s", f"{out_w}x{out_h}",
            "-pix_fmt", "bgr24",
            "-r", str(fps),
            "-i", "-"
        ]
        
        if has_audio:
            ffmpeg_cmd.extend(["-i", str(temp_audio), "-c:a", "copy"])
            
        ffmpeg_cmd.extend([
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-crf", str(QUALIDADE_CRF),
            "-preset", "fast",
            "-movflags", "+faststart",
            str(out_path)
        ])
        
        ffmpeg_proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)
        
        t0 = time.perf_counter()
        try:
            with tqdm(total=total_frames, desc=f"Upscaling ({active_model_key})", unit="frame") as pbar:
                while True:
                    ret, frame = cap.read()
                    if not ret or frame is None:
                        break
                        
                    t_frame = torch.from_numpy(frame).permute(2, 0, 1).unsqueeze(0).float() / 255.0
                    model_dtype = globals().get('UPSCALE_MODEL_DTYPE', torch.float32)
                    t_frame = t_frame.to(device=device, dtype=model_dtype)
                        
                    with torch.no_grad():
                        sr_tensor = upscale_model(t_frame)
                        
                    sr_frame = (sr_tensor.squeeze(0).permute(1, 2, 0).clamp(0, 1) * 255.0).byte().cpu().numpy()
                    
                    if (sr_frame.shape[1], sr_frame.shape[0]) != (out_w, out_h):
                        sr_frame = cv2.resize(sr_frame, (out_w, out_h), interpolation=cv2.INTER_AREA if sr_frame.shape[1] > out_w else cv2.INTER_LANCZOS4)
                        
                    ffmpeg_proc.stdin.write(sr_frame.tobytes())
                    pbar.update(1)
        finally:
            cap.release()
            if ffmpeg_proc.stdin:
                ffmpeg_proc.stdin.close()
            ffmpeg_proc.wait()
            if temp_audio.is_file():
                temp_audio.unlink(missing_ok=True)
                
        elapsed = time.perf_counter() - t0
        print(f"\n✅ Concluído em {elapsed:.1f}s ({total_frames / max(elapsed, 0.001):.1f} fps)")
        print(f"📁 Salvo em: {out_path} ({out_path.stat().st_size / (1024**2):.1f} MiB)")
        
        if v_idx == len(videos_to_process) and out_path.stat().st_size < 35 * 1024 * 1024:
            try:
                _data_url = "data:video/mp4;base64," + b64encode(out_path.read_bytes()).decode()
                display(HTML(f'''
                <div style="text-align: center; margin-top: 15px;">
                    <h4>🎬 Pré-visualização ({active_model_key}): {out_filename}</h4>
                    <video width="{min(out_w, 860)}" controls autoplay loop style="border-radius: 8px; max-width: 100%;">
                        <source src="{_data_url}" type="video/mp4">
                    </video>
                </div>
                '''))
            except Exception:
                pass

    print("\n🎉 Todos os vídeos da fila foram upscaled com sucesso!")
    print(f"📂 Verifique sua pasta no Drive: {OUTPUT_DIR}")
